# Rust / 병렬·튜닝 확장 리포트

베이스라인 요약·표·그래프는 **`rust_speed_efficiency_report.ipynb`** 를 유지합니다. 본 노트북은 **추가로 속도를 끌어올릴 수 있는 레버**를 정리하고, 재현 가능한 **스윕 벤치**(Rayon 스레드, Rust 배치 크기)를 실행합니다.

**핵심 요약:** QDF Rust 전처리는 이미 **`qdf_io` + Rayon** 으로 분자 배치 내부가 병렬입니다. NumPy 경로는 SciPy/OpenBLAS가 **기본 단일 스레드에 가깝게** 도는 경우가 많아, “Python에서 멀티프로세스로 분자 나누기” 없이는 추가 병렬 이득이 제한적입니다. 학습 단계에서는 **`DataLoader` `num_workers`** 가 별도 축입니다.

## 1. 병렬이 걸리는 위치 (정리)

| 구간 | 병렬 방식 | 조절 노브 |
|------|-----------|-----------|
| `preprocess.py` **numpy** | 보통 분자 루프는 순차 + BLAS 일부 멀티 | `OMP_NUM_THREADS` / `OPENBLAS_NUM_THREADS` 등(환경·빌드 의존) |
| `preprocess.py` **rust** | `preprocess_batch_rust` 내부 **Rayon** | **`RAYON_NUM_THREADS`** (기본: 논리 코어 수) |
| Rust 배치 크기 | 배치마다 한 번의 네이티브 호출 | **`--rust-batch-size`** (`train/preprocess.py`, `profile_preprocess.py`) |
| **학습** 데이터 적재 | PyTorch `DataLoader` | **`num_workers`** (`train.py` 위치 인자) |
| LCAO `pad` Rust | CPU 커널; GPU 스트림과 경합 가능 | `--pad-impl` (연구노트: 당시 환경에선 비추천) |

DGCL pretrain end-to-end는 연구노트대로 **PyG Batch 분해/재조립** 병목이 크므로, 이 노트의 스윕은 **QDF preprocess + (선택) views** 에 집중합니다.

In [ ]:
from __future__ import annotations

import os
import re
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
for p in [REPO, *REPO.parents]:
    if (p / "연구노트.md").is_file():
        REPO = p
        break

PYTHON = sys.executable
PROFILE_PRE = REPO / "QuantumDeepField_molecule" / "bench" / "profile_preprocess.py"

# 스윕 부하 (True면 빠른 스모크)
SMOKE = True
PRE_LIMIT = 80 if SMOKE else 600

# True: Rayon 스레드 수 스윕 / rust 배치 크기 스윕 실행 (시간 증가)
RUN_RAYON_SWEEP = True
RUN_RUST_BATCH_SWEEP = True

print("REPO =", REPO)
print("PRE_LIMIT =", PRE_LIMIT, "| SMOKE =", SMOKE)

In [ ]:
def run_cmd(args: list[str], *, cwd: Path | None = None, extra_env: dict[str, str] | None = None):
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    return subprocess.run(
        [PYTHON, *args],
        cwd=str(cwd or REPO),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
    )


def parse_wall(stdout: str) -> float | None:
    m = re.search(r"wall time\s+:\s+([\d.]+)\s+s", stdout)
    return float(m.group(1)) if m else None


def bench_rust(*, rust_batch: int, rayon_threads: str | None = None) -> tuple[float | None, int]:
    extra = {"RAYON_NUM_THREADS": rayon_threads} if rayon_threads is not None else {}
    cp = run_cmd(
        [
            str(PROFILE_PRE),
            "--dataset",
            "QM9under14atoms_atomizationenergy_eV",
            "--split",
            "train",
            "--limit",
            str(PRE_LIMIT),
            "--no-save",
            "--backend",
            "rust",
            "--rust-batch-size",
            str(rust_batch),
        ],
        extra_env=extra,
    )
    return parse_wall(cp.stdout or ""), cp.returncode


if not PROFILE_PRE.is_file():
    raise FileNotFoundError(PROFILE_PRE)

## 2. `RAYON_NUM_THREADS` 스윕 (Rust 전처리만)

같은 `PRE_LIMIT`·`--rust-batch-size 64` 로 **Rayon 스레드 상한**만 바꿉니다. 코어 수보다 크게 잡아도 이득이 없거나 오히려 느려질 수 있습니다.

In [ ]:
import pandas as pd

rayon_rows: list[dict] = []
if RUN_RAYON_SWEEP:
    logical = os.cpu_count() or 8
    candidates = sorted({1, 2, 4, max(1, logical // 2), logical, logical * 2})
    for t in candidates:
        wall, rc = bench_rust(rust_batch=64, rayon_threads=str(t))
        rayon_rows.append({"RAYON_NUM_THREADS": t, "wall_s": wall, "rc": rc})
        print(f"RAYON_NUM_THREADS={t:>3}  wall_s={wall}  rc={rc}")
    df_rayon = pd.DataFrame(rayon_rows)
    try:
        display(df_rayon)
    except NameError:
        print(df_rayon.to_string(index=False))
else:
    print("SKIP: RUN_RAYON_SWEEP = False")

## 3. `--rust-batch-size` 스윕

배치가 크면 **네이티브 호출 횟수**는 줄지만, 배치당 메모리·스케줄 오버헤드 트레이드오프가 있습니다. 데이터셋/머신마다 최적값이 다릅니다.

In [ ]:
batch_sizes = [1, 8, 16, 32, 64, 128] if not SMOKE else [1, 16, 64, 128]
batch_rows: list[dict] = []

if RUN_RUST_BATCH_SWEEP:
    for bs in batch_sizes:
        wall, rc = bench_rust(rust_batch=bs, rayon_threads=None)
        batch_rows.append({"rust_batch_size": bs, "wall_s": wall, "rc": rc})
        print(f"rust_batch_size={bs:>4}  wall_s={wall}  rc={rc}")
    df_bs = pd.DataFrame(batch_rows)
    try:
        display(df_bs)
    except NameError:
        print(df_bs.to_string(index=False))

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(df_bs["rust_batch_size"], df_bs["wall_s"], "o-", color="#55A868")
    ax.set_xlabel("rust_batch_size")
    ax.set_ylabel("wall (s)")
    ax.set_title(f"Rust preprocess wall vs batch (n={PRE_LIMIT}, --no-save)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("SKIP: RUN_RUST_BATCH_SWEEP = False")

## 4. 학습 측 병렬 (`num_workers`)

`train/train.py` 는 위치 인자로 **`num_workers`** 를 받습니다. `train/train.sh` 에는 Windows·RDKit 이슈를 피하려 **`num_workers=0`** 기본 주석이 있으나, QDF 스택에서는 **4 전후**로 올려 IO·CPU 전처리를 겹치는 경우가 많습니다. **shard + `MyDatasetShard`** 사용 시 Windows에서는 예전처럼 worker pickle 문제가 나지 않도록 `dataset_shard` 쪽이 정리되어 있다는 전제(연구노트)에서 시도하세요.

전처리와 달리 **GPU/XPU가 지배적**이면 `num_workers` 이득은 상한이 낮습니다.

## 5. “더 빠르게” 우선순위 (실무)

1. **전처리 한 번만** 돌리고 **`--output-format shard`** 로 IO·파일 수를 줄인다.  
2. Rust 경로에서 **`--rust-batch-size`** 를 스윕해 최소 wall 을 고른다(본 노트 §3).  
3. **`RAYON_NUM_THREADS`** 를 물리 코어 수 근처로 맞춘다(§2).  
4. 학습에서 **`num_workers`** 를 0→2→4 로 올려 본다(메모리·Windows 안정성 확인).  
5. NumPy 전처리를 계속 쓸 경우에만 **OpenBLAS/MKL 스레드** 환경변수 튜닝을 검토(다른 작업과 CPU 경합 주의).  
6. **멀티프로세스로 분자 파일을 샤드 나눠 전처리**하는 것은 스크립트 미구현 상태 — 레포에 들어가면 운영 복잡도만큼 이득이 큰 편.

---

원본 벤치 요약: `rust_speed_efficiency_report.ipynb`